In [ ]:
from os import environ, makedirs, path

import pandas as pd
from dotenv import load_dotenv
from unidecode import unidecode

In [ ]:
load_dotenv()

# Atualização de 2026 - Mortalidade Materna e Mortalidade Infantil

Para o ano de 2026, o GT Saúde decidiu adicionar dois painéis com dados específicos de mortalidade, um para mortalidade materna e outro para mortalidade infantil. Esses temas já haviam sido sugeridos pelos especialistas que participaram em 2025, e também foram trabalhados pela auditoria do TCMSP, resultando inclusive em um evento em parceria com o OPP no ano de 2026.

Agora, precisamos adicionar esses dados padronizados no formato adequado ao Qlik Sense para que sejam incorporados ao DataSP.

Os dados são coeficientes semelhantes ao de mortalidade geral, mas com recortes específicos. De todo modo, tratam-se de coeficientes ponderados. No caso de mortalidade materna, temos o seguinte coeficiente:

$$ \text{Razão de mortalidade materna} = \frac{\text{Número de óbitos de mulheres por causas e condições consideradas de morte materna}}{\text{Número de nascidos vivos}} \times 100.000 $$

E, no caso de mortalidade infantil, temos:

$$ \text{Taxa de mortalidade infantil} = \frac{\text{Número de óbitos menores de 1 ano de idade}}{\text{Número de nascidos vivos}} \times 1.000 $$

Para que possamos recalcular esses indicadores de acordo com a seleção de Subprefeituras feito dinamicamente pelo usuário, precisamos obter o numerador e o denomidador deles e agregá-los dinamicamente de acordo com a seleção.

Para isso, existem duas alternativas possíveis. Uma delas é obter a razão (ou taxa) de mortalidade já calculada para cada Subprefeitura e estimar o número de óbitos com base no número de nascidos vivos. Outra forma seria obter os componentes da razão (taxa) e calculá-lo para cada Subprefeitura. A primeira maneira é preferida, porque garante que os indicadores reflitam exatamente os valores disponíveis em bases de dados oficiais. Entretanto, esses valores existem apenas para o ano de 2024. Para o ano de 2025, estão disponíveis apenas dados provisórios de mortalidade geral, de modo que o cálculo da razão (taxa) é possível de modo provisório, ainda que não esteja disponível em bases oficiais.

Portanto, as duas maneiras serão utilizadas: para o ano de 2024, serão utilizados os valores diponíveis em bases oficiais e o número de óbitos será estimado com base no número de nascidos vivos; para o ano de 2025, a razão (taxa) será calculada com base nos dados de mortalidade geral e nascidos vivos.

# Extração dos dados

## Mortalidade Materna

Esta razão será obtida do ObservaSampa, mas o download está apresentando um erro no momento. Os dados foram extraídos e corrigidos manualmente e estão disponíveis na pasta de dados do projeto

### 2024

In [ ]:
data_input_dir = path.join('data', 'input', 'saude')

In [ ]:
file_name_materna = 'mortalidade_materna_2024.csv'
df_materna_24 = pd.read_csv(path.join(data_input_dir, file_name_materna), sep=';', encoding='latin1', decimal=',')
df_materna_24

### 2025

Para 2025, obteremos os dados de mortalidade geral filtrando para mulheres com causas relacionadas ao Capítulo XV. Gravidez parto e puerpério do CID10. Esses dados também são extraídos manualmente, desta vez do TabNet, e adicionados a pasta de dados deste projeto.

In [ ]:
file_name_obitos_materna_25 = 'A002518192_29_143_95.csv'
df_obitos_materna_25 = pd.read_csv(path.join(data_input_dir, file_name_obitos_materna_25),
                                   sep=';', encoding='latin1', decimal=',', skiprows=5)
df_obitos_materna_25

Vamos remover as linhas finais com informações complementares sobre os dados

In [ ]:
df_obitos_materna_25 = df_obitos_materna_25[:-6]
df_obitos_materna_25

## Mortalidade Infantil

### 2024

Neste caso, tanto a taxa, quanto os dados provisórios de óbitos serão obtidos do TabNet. Vamos começar pela taxa de mortalidade infantil.

In [ ]:
file_name_infantil_24 = 'A003317192_29_141_165.csv'
df_infantil_24 = pd.read_csv(path.join(data_input_dir, file_name_infantil_24),
                                         sep=';', encoding='latin1', decimal=',', skiprows=3)
df_infantil_24

Novamente, vamos remover as linhas de totalização e informações adicionais.

In [ ]:
df_infantil_24 = df_infantil_24[:-8]
df_infantil_24

### 2025

In [ ]:
file_name_obitos_infantil_25 = 'A003858192_29_141_165.csv'
df_obitos_infantil_25 = pd.read_csv(path.join(data_input_dir, file_name_obitos_infantil_25),
                                    sep=';', encoding='latin1', decimal=',', skiprows=4)
df_obitos_infantil_25 = df_obitos_infantil_25[:-6]
df_obitos_infantil_25

## Nascidos Vivos

Os dados de nascidos vivos também são obtidos do TabNet.

In [ ]:
file_name_nv = 'A004130192_29_141_165.csv'
df_nv = pd.read_csv(path.join(data_input_dir, file_name_nv),
                    sep=';', encoding='latin1', decimal=',', skiprows=3)
df_nv = df_nv[:-3]
df_nv

Nesse caso, temos também que excluir a coluna de totalização.

In [ ]:
df_nv = df_nv.drop(columns=['Total'])
df_nv

## Subprefeituras

Para que possamos unir os dados aqui e no Qlik precisamos padronizar o nome das Subprefeituras para utilizar como chave. Para isso, vamos carregar o arquivo de subprefeituras utilizado no Qlik.

In [ ]:
url_subs = environ.get('CSV_SUBPREFEITURAS_QLIK')
df_subs = pd.read_csv(url_subs)
df_subs

# Transformação dos dados

## Padronizando os nomes de subprefeituras

In [ ]:
df_subs_qlik = df_subs['sub.NOME'].sort_values().tolist()
df_subs_qlik

### Mortalidade Materna

In [ ]:
subs_materna_24 = (
    df_materna_24[df_materna_24['nivel_regional'] == 'Subprefeitura']
    .assign(Subprefeitura=df_materna_24['região'].apply(unidecode))
    .sort_values('Subprefeitura')
    ['região']
    .tolist()
)
subs_materna_24

In [ ]:
map_materna_24 = dict(zip(subs_materna_24, df_subs_qlik))
map_materna_24

In [ ]:
df_materna_24['sub.NOME'] = df_materna_24['região'].map(map_materna_24)
df_materna_24

In [ ]:
subs_materna_25 = (
    df_obitos_materna_25
    .assign(Subprefeitura=df_obitos_materna_25['Subprefeitura residência'].apply(unidecode))
    .sort_values('Subprefeitura')
    ['Subprefeitura residência']
    .tolist()
)
subs_materna_25.remove('Ignorado')
subs_materna_25

In [ ]:
map_materna_25 = dict(zip(subs_materna_25, df_subs_qlik))
map_materna_25

In [ ]:
df_obitos_materna_25.loc[:, 'sub.NOME'] = df_obitos_materna_25['Subprefeitura residência'].map(map_materna_25)
df_obitos_materna_25

### Mortalidade Infantil

In [ ]:
subs_infantil_24 = (
    df_infantil_24
    .assign(Subprefeitura=df_infantil_24['Subprefeitura resid'].apply(unidecode))
    .sort_values('Subprefeitura')
    ['Subprefeitura resid']
    .tolist()
)
subs_infantil_24.remove('Ignorado')
subs_infantil_24

In [ ]:
map_infantil_24 = dict(zip(subs_infantil_24, df_subs_qlik))
map_infantil_24

In [ ]:
df_infantil_24.loc[:, 'sub.NOME'] = df_infantil_24.loc[:, 'Subprefeitura resid'].map(map_infantil_24)
df_infantil_24

In [ ]:
subs_infantil_25 = (
    df_obitos_infantil_25
    .assign(Subprefeitura=df_obitos_infantil_25['Subprefeitura residência'].apply(unidecode))
    .sort_values('Subprefeitura')
    ['Subprefeitura residência']
    .tolist()
)
subs_infantil_25.remove('Ignorado')
subs_infantil_25

In [ ]:
map_infantil_25 = dict(zip(subs_infantil_25, df_subs_qlik))
map_infantil_25

In [ ]:
df_obitos_infantil_25.loc[:, 'sub.NOME'] = df_obitos_infantil_25.loc[:, 'Subprefeitura residência'].map(map_infantil_25)
df_obitos_infantil_25

### Nascidos Vivos

In [ ]:
subs_nv = (
    df_nv
    .assign(Subprefeitura=df_nv['SubPrefeitura residência'].apply(unidecode))
    .sort_values('Subprefeitura')
    ['SubPrefeitura residência']
    .tolist()
)
subs_nv.remove('Ignorado')
subs_nv

In [ ]:
map_nv = dict(zip(subs_nv, df_subs_qlik))
map_nv

In [ ]:
df_nv.loc[:, 'sub.NOME'] = df_nv.loc[:, 'SubPrefeitura residência'].map(map_nv)
df_nv

## Mortalidade Materna

Primeiro, vamos manter apenas o nível de subprefeitura e ano de 2024 para os dados vindos do ObservaSampa.

In [ ]:
df_materna_24 = df_materna_24[df_materna_24['nivel_regional'] == 'Subprefeitura']
df_materna_24 = df_materna_24[['região', 'sub.NOME', '2024']]
df_materna_24 = df_materna_24.rename(columns={'região': 'Subprefeitura'})
df_materna_24

Agora, vamos transformar o ano em uma coluna.

In [ ]:
df_materna_24 = pd.melt(df_materna_24,
                        id_vars=['Subprefeitura', 'sub.NOME'],
                        var_name='Ano',
                        value_name='Razão de mortalidade materna')
df_materna_24

Agora, vamos adicionar a coluna de nascidos vivos para poder calcular o coeficiente.

In [ ]:
df_nv  = pd.melt(df_nv,
                 id_vars=['SubPrefeitura residência', 'sub.NOME'],
                 var_name='Ano',
                 value_name='Número de nascidos vivos')
df_nv

In [ ]:
df_materna = df_materna_24.merge(
    df_nv[['sub.NOME', 'Ano', 'Número de nascidos vivos']],
    how='left',
    left_on=['sub.NOME', 'Ano'],
    right_on=['sub.NOME', 'Ano']
)
df_materna

In [ ]:
df_materna['Óbitos_Residentes_MSP'] = df_materna['Razão de mortalidade materna'] * df_materna['Número de nascidos vivos'] / 100000
df_materna = df_materna.sort_values(['sub.NOME', 'Ano']).reset_index(drop=True)
df_materna

In [ ]:
df_materna_25 = (
    df_obitos_materna_25
    .loc[:, ['sub.NOME', 'Óbitos_Residentes_MSP']]
    .assign(Ano='2025')
    .merge(
        df_nv[['sub.NOME', 'Ano', 'Número de nascidos vivos']],
        how='left',
        left_on=['sub.NOME', 'Ano'],
        right_on=['sub.NOME', 'Ano']
    )
)
df_materna_25

In [ ]:
df_materna_25['Óbitos_Residentes_MSP'] = df_materna_25['Óbitos_Residentes_MSP'].str.replace('-', '0')
df_materna_25['Óbitos_Residentes_MSP'] = df_materna_25['Óbitos_Residentes_MSP'].astype(float)
df_materna_25

In [ ]:
df_materna_25['Razão de mortalidade materna'] = df_materna_25['Óbitos_Residentes_MSP'] / df_materna_25['Número de nascidos vivos'] * 100000
df_materna_25

In [ ]:
df_materna = pd.concat([df_materna, df_materna_25], ignore_index=True)
df_materna = df_materna.drop(columns=['Subprefeitura'])
df_materna

In [ ]:
df_materna.groupby('Ano')['Óbitos_Residentes_MSP'].sum()

## Mortalidade infantil

In [ ]:
df_infantil = (
    pd.concat([
    df_infantil_24.rename(columns={'Subprefeitura resid': 'Subprefeitura'})
    .assign(Ano='2024'),
    df_obitos_infantil_25.rename(columns={'Subprefeitura residência': 'Subprefeitura'})
    .assign(Ano='2025')
    ])
    .merge(df_nv[['sub.NOME', 'Ano', 'Número de nascidos vivos']],
           how='left',
           left_on=['sub.NOME', 'Ano'],
           right_on=['sub.NOME', 'Ano'])
)
df_infantil

In [ ]:
df_infantil.loc[df_infantil['Óbitos_Residentes_MSP'].isna(), 'Óbitos_Residentes_MSP'] = (
    df_infantil['Coef_mort_infantil'] * df_infantil['Número de nascidos vivos'] / 1000
).round(0)
df_infantil

In [ ]:
df_infantil.loc[df_infantil['Coef_mort_infantil'].isna(), 'Coef_mort_infantil'] = (
    df_infantil['Óbitos_Residentes_MSP'] * 1000 / df_infantil['Número de nascidos vivos']
).round(2)
df_infantil

In [ ]:
df_infantil.groupby('Ano')['Óbitos_Residentes_MSP'].sum()

# Armazenamento dos dados

Finalmente, salvamos os arquivos como csv para utilizarmos no Qlik Sense.

In [ ]:
base_path = path.join('data_output', 'saude')

if not path.exists(base_path):
    makedirs(base_path)

for name, df in [('mortalidade-materna', df_materna),
                 ('mortalidade-infantil', df_infantil),]:
    
    filepath = path.join(base_path, f'{name}.csv')

    df.to_csv(filepath,
              index=False,
              sep=';',
              decimal=',')